In [1]:
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
from IPython.display import HTML, display
from train import train_agent
import yaml
from env import CameraEnv

def render(env, agent, steps=200):
    s, _ = env.reset()

    for _ in range(steps):
        a = agent.act(s)
        s, _, d, _, _ = env.step(a)

        plt.figure(figsize=(6, 6))
        plt.xlim(0, 1)
        plt.ylim(0, 1)
        plt.scatter([env.obj_pos[0]], [env.obj_pos[1]], c='black', s=100)  # объект
        plt.scatter([env.cam_pos[0]], [env.cam_pos[1]], c='red', s=100)    # камера
        plt.gca().set_aspect('equal')
        plt.pause(0.01)
        plt.close()

        if d:
            break

def render_animation(env, agent, steps=200, interval=50, save_path=None, show=True):
    s, _ = env.reset()
    obj_positions = []
    cam_positions = []

    for _ in range(steps):
        a = agent.act(s)
        s, _, d, _, _ = env.step(a)
        obj_positions.append((float(env.obj_pos[0]), float(env.obj_pos[1])))
        cam_positions.append((float(env.cam_pos[0]), float(env.cam_pos[1])))
        if d:
            break

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')

    obj_scatter = ax.scatter([], [], c='black', s=100)  # объект
    cam_scatter = ax.scatter([], [], c='red', s=100)    # камера

    empty = np.empty((0, 2))

    def init():
        obj_scatter.set_offsets(empty)
        cam_scatter.set_offsets(empty)
        return obj_scatter, cam_scatter

    def update(i):
        obj_scatter.set_offsets([obj_positions[i]])
        cam_scatter.set_offsets([cam_positions[i]])
        return obj_scatter, cam_scatter

    anim = animation.FuncAnimation(
        fig,
        update,
        init_func=init,
        frames=len(obj_positions),
        interval=interval,
        blit=True,
    )

    if save_path:
        fps = 1000 / interval if interval > 0 else 20
        anim.save(save_path, fps=fps)

    if show:
        try:
            display(HTML(anim.to_jshtml()))
        except Exception:
            pass

    return anim


In [3]:
agent = train_agent()
env = CameraEnv(yaml.safe_load(open("config.yaml"))["env"])
render_animation(env, agent)

ep 0 reward -109.50
ep 1 reward -78.39
ep 2 reward -63.22


KeyboardInterrupt: 